In [154]:
#THIS IS ALL AI GENERATED!


"""Load tiepoint files and return one merged DataFrame per selected core."""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable

import os
import numpy as np
import pandas as pd

os.chdir('/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/all_tiepoints')

def _normalize_core_list(cores: Iterable[str]) -> set[str]:
    return {str(core).strip().upper() for core in cores if str(core).strip()}


def _extract_pair_from_folder(folder_name: str) -> tuple[str, str] | None:
    if "-" not in folder_name:
        return None
    parts = folder_name.split("-")
    if len(parts) != 2:
        return None
    return parts[0].upper(), parts[1].upper()


def _read_tiepoint_file(path: Path) -> pd.DataFrame:
    # Files are tab-separated in practice, but whitespace parsing is robust.
    df = pd.read_csv(path, sep=r"\s+", engine="python", comment="#")
    normalized = {col: str(col).strip().lower() for col in df.columns}
    df = df.rename(columns=normalized)

    required = {"depth1", "depth2"}
    if not required.issubset(df.columns):
        # Fallback for files without header rows.
        df = pd.read_csv(
            path,
            sep=r"\s+",
            engine="python",
            comment="#",
            header=None,
            names=["depth1", "depth2", "reference"],
        )

    if "reference" not in df.columns:
        df["reference"] = pd.NA

    return df[["depth1", "depth2", "reference"]].copy()


def _merge_pair_rows(
    load_data: pd.DataFrame,
    pair: str,
    num_files: int,
    error_margin: float,
    verbose: bool = True,
) -> pd.DataFrame:
    """Apply the same duplicate-merging logic used in pair-processing code."""
    if load_data.empty:
        return load_data

    load_data = load_data.copy().reset_index(drop=True)
    load_data["depth1"] = pd.to_numeric(load_data["depth1"], errors="coerce")
    load_data["depth2"] = pd.to_numeric(load_data["depth2"], errors="coerce")
    load_data = load_data.dropna(subset=["depth1", "depth2"]).reset_index(drop=True)

    drop_rows = []
    drop_rows_merge = set()
    new_merged_rows = []

    for idx, row in load_data.iterrows():
        mask1 = (load_data["depth1"] - row["depth1"]).abs() <= error_margin
        mask1.at[idx] = False
        mask2 = (load_data["depth2"] - row["depth2"]).abs() <= error_margin
        mask2.at[idx] = False

        close_idxs = load_data.index[mask1 & mask2]
        if len(close_idxs) > 0:
            refs = [load_data.at[idx, "reference"]] + [load_data.at[i, "reference"] for i in close_idxs]
            merged_ref = "; ".join(str(r) for r in refs if pd.notna(r))

            depth1_vals = [load_data.at[idx, "depth1"]] + [load_data.at[i, "depth1"] for i in close_idxs]
            merged_depth1 = np.round(np.mean(depth1_vals), 4)

            depth2_vals = [load_data.at[idx, "depth2"]] + [load_data.at[i, "depth2"] for i in close_idxs]
            merged_depth2 = np.round(np.mean(depth2_vals), 4)

            merged_row = {
                "reference": merged_ref,
                "depth1": merged_depth1,
                "depth2": merged_depth2,
            }

            if "source_file" in load_data.columns:
                srcs = [load_data.at[idx, "source_file"]] + [load_data.at[i, "source_file"] for i in close_idxs]
                merged_row["source_file"] = "; ".join(str(s) for s in dict.fromkeys(srcs) if pd.notna(s))

            new_merged_rows.append(merged_row)
            drop_rows_merge.add(idx)

            for i in close_idxs:
                drop_rows.append(i)
                if verbose and drop_rows.count(i) >= num_files:
                    print(
                        f"WARNING: Row {load_data.at[i, 'depth1']} | {load_data.at[i, 'depth2']} for {pair}. "
                        f"Reference {load_data.at[i, 'reference']}."
                    )
                    print(
                        f"Called by row {load_data.at[idx, 'depth1']} | {load_data.at[idx, 'depth2']} "
                        f"from reference {load_data.at[idx, 'reference']}."
                    )

    drop_rows = set(drop_rows).union(drop_rows_merge)
    load_data = load_data.drop(index=drop_rows).reset_index(drop=True)

    merged_df = pd.DataFrame(new_merged_rows)
    if "source_file" in load_data.columns and "source_file" not in merged_df.columns and not merged_df.empty:
        merged_df["source_file"] = pd.NA

    load_data = pd.concat([load_data, merged_df], ignore_index=True)
    load_data.drop_duplicates(subset=["depth1", "depth2"], inplace=True)
    load_data = load_data.sort_values(by=["depth1"]).reset_index(drop=True)

    return load_data


def load_tiepoints_for_cores(
    cores: Iterable[str],
    base_dir: str | Path | None = None,
    match_all: bool = False,
    error_margin: float = 0.0,
    verbose: bool = True,
) -> Dict[str, pd.DataFrame]:
    """Return one merged DataFrame per selected core.

    Each per-core table has:
    - a depth column named after the main core (e.g., NG1)
    - paired_core_depths
    - paired_core_name
    plus reference/source_file/pair metadata.
    """
    selected = _normalize_core_list(cores)
    if not selected:
        return {}

    root = Path(base_dir).expanduser().resolve() if base_dir is not None else Path.cwd()
    tables_by_core: Dict[str, list[pd.DataFrame]] = {core: [] for core in sorted(selected)}
    pair_dfs: Dict[str, list[pd.DataFrame]] = {}

    for txt_path in sorted(root.rglob("*.txt")):
        pair = _extract_pair_from_folder(txt_path.parent.name)
        if pair is None:
            continue

        core1, core2 = pair
        pair_set = {core1, core2}

        if match_all:
            keep = pair_set.issubset(selected)
        else:
            keep = bool(pair_set & selected)

        if not keep:
            continue

        pair_name = f"{core1}-{core2}"
        df = _read_tiepoint_file(txt_path)
        df["source_file"] = f"{txt_path.parent.name}/{txt_path.name}"
        pair_dfs.setdefault(pair_name, []).append(df)

    for pair_name, dfs in pair_dfs.items():
        core1, core2 = pair_name.split("-")
        merged = _merge_pair_rows(
            load_data=pd.concat(dfs, ignore_index=True),
            pair=pair_name,
            num_files=len(dfs),
            error_margin=error_margin,
            verbose=verbose,
        )
        merged["pair"] = pair_name

        if verbose:
            print(
                f"Processed pair {pair_name}, total points after merging: {len(merged)} "
                f"({sum(len(df) for df in dfs)} original total rows)"
            )

        if core1 in selected:
            core1_view = pd.DataFrame({
                core1: merged["depth1"],
                "paired_core_depths": merged["depth2"],
                "paired_core_name": core2,
                "reference": merged["reference"],
                "source_file": merged.get("source_file", pd.Series([pd.NA] * len(merged))),
                "pair": merged["pair"],
            })
            tables_by_core[core1].append(core1_view)

        if core2 in selected:
            core2_view = pd.DataFrame({
                core2: merged["depth2"],
                "paired_core_depths": merged["depth1"],
                "paired_core_name": core1,
                "reference": merged["reference"],
                "source_file": merged.get("source_file", pd.Series([pd.NA] * len(merged))),
                "pair": merged["pair"],
            })
            tables_by_core[core2].append(core2_view)

    out: Dict[str, pd.DataFrame] = {}
    for core, parts in tables_by_core.items():
        cols = [core, "paired_core_depths", "paired_core_name", "reference", "source_file", "pair"]
        if parts:
            combined = pd.concat(parts, ignore_index=True)
            combined = combined.sort_values(by=[core]).reset_index(drop=True)
            out[core] = combined[cols]
        else:
            out[core] = pd.DataFrame(columns=cols)

    return out


load_cores = ["NG1", "NG2", "GRIP", "WDC"]
tiepoint_tables = load_tiepoints_for_cores(load_cores, error_margin=0.15)
print(f"Built {len(tiepoint_tables)} per-core tables for: {load_cores}")
for core in load_cores:
    table = tiepoint_tables.get(core, pd.DataFrame())
    n_files = table["source_file"].nunique() if "source_file" in table.columns else 0
    print(f"{core}: {len(table)} rows from {n_files} files")

print("\nPreview (WDC):")
print(tiepoint_tables["WDC"].head())

Processed pair DF-WDC, total points after merging: 587 (1078 original total rows)
Processed pair EDC-WDC, total points after merging: 636 (897 original total rows)
Processed pair EDML-WDC, total points after merging: 974 (1315 original total rows)
Called by row 1663.938 | 1609.565 from reference Seierstad2014_Rasmussen2014.
Processed pair GISP2-GRIP, total points after merging: 823 (918 original total rows)
Processed pair GISP2-NG1, total points after merging: 290 (290 original total rows)
Processed pair GISP2-NG2, total points after merging: 614 (725 original total rows)
Processed pair GISP2-WDC, total points after merging: 286 (299 original total rows)
Processed pair GRIP-DF, total points after merging: 107 (107 original total rows)
Processed pair GRIP-EDC, total points after merging: 119 (119 original total rows)
Processed pair GRIP-EDML, total points after merging: 127 (127 original total rows)
Processed pair GRIP-NEEM, total points after merging: 396 (396 original total rows)
Proc

In [155]:
gicc05_chron = pd.read_excel(
    '/Users/quinnmackay/Documents/GitHub/BICC/Data Storage/Chronologies/gicc05_time_scale_excel.xlsx',
    skiprows=40,
    usecols=[0, 2, 3, 4],
    names=['age b2k', 'depth_NG1', 'depth_NG2', 'depth_GRIP'],
)

wdc_chron = pd.read_excel(
    '/Users/quinnmackay/Documents/GitHub/BICC/Data Storage/Chronologies/WD2014 Chronology.xlsx',
    skiprows=35,
    sheet_name=4,
    usecols=[0, 1],
    names=['age b2k', 'depth_WDC'],
)
wdc_chron['age b2k'] = wdc_chron['age b2k'] + 50

def _interp_ages(depth_values: pd.Series, depth_grid: pd.Series, age_grid: pd.Series) -> np.ndarray:
    """Interpolate ages using a cleaned monotonic depth-age grid."""
    grid = pd.DataFrame({'depth': depth_grid, 'age': age_grid}).dropna().sort_values('depth')
    return np.interp(depth_values, grid['depth'], grid['age'])

for core in ['NG1', 'NG2', 'GRIP']:
    table = tiepoint_tables[core].copy()
    depth_col = core
    valid = table[depth_col].notna()
    table.loc[valid, f'{core}_age'] = np.round(
        _interp_ages(
            table.loc[valid, depth_col],
            gicc05_chron[f'depth_{core}'],
            gicc05_chron['age b2k'],
        ),
        3,
    )
    tiepoint_tables[core] = table

table = tiepoint_tables['WDC'].copy()
valid = table['WDC'].notna()
table.loc[valid, 'WDC_age'] = np.round(
    _interp_ages(
        table.loc[valid, 'WDC'],
        wdc_chron['depth_WDC'],
        wdc_chron['age b2k'],
    ),
    3,
 )
tiepoint_tables['WDC'] = table

In [144]:
tiepoint_tables['GRIP']

,GRIP,paired_core_depths,paired_core_name,reference,source_file,pair,GRIP_age
0,13.003,19.839,NEEM,GICC21,GRIP-NEEM/GICC21_GRIP-NEEM.txt,GRIP-NEEM,38.408
1,13.003,13.735,NG1,GICC21,GRIP-NG1/GICC21_GRIP-NG1.txt,GRIP-NG1,38.408
2,13.003,14.100,NG2,GICC21,GRIP-NG2/GICC21_GRIP-NG2.txt,GRIP-NG2,38.408
3,17.342,17.199,NG1,GICC21,GRIP-NG1/GICC21_GRIP-NG1.txt,GRIP-NG1,49.312
4,17.342,23.419,NEEM,GICC21,GRIP-NEEM/GICC21_GRIP-NEEM.txt,GRIP-NEEM,49.312
...,...,...,...,...,...,...,...
3245,2753.960,2756.880,GISP2,Svensson_Links,GISP2-GRIP/Svensson_Links_GISP2-GRIP.txt,GISP2-GRIP,NaN
3246,2753.960,1512.070,DF,SvenssonLinks,GRIP-DF/SvenssonLinks_GRIP-DF.txt,GRIP-DF,NaN
3247,2753.960,2905.020,NG2,Svensson_Links,GRIP-NG2/Svensson_Links_GRIP-NG2.txt,GRIP-NG2,NaN
3248,2753.960,2200.410,EDML,SvenssonLinks,GRIP-EDML/SvenssonLinks_GRIP-EDML.txt,GRIP-EDML,NaN


In [157]:
# specific depth ranges

for core in ['NG1', 'NG2', 'GRIP', 'WDC']:
    table = tiepoint_tables[core].copy()
    age_col = f'{core}_age'
    if core == 'WDC':
        valid = table[age_col].notna() & (table[age_col] - 50 >= 9.50 * 1000) & (table[age_col] - 50 <= 9.60 * 1000)
    else:
        valid = table[age_col].notna() & (table[age_col] >= 9.58 * 1000) & (table[age_col] <= 9.7 * 1000)
    table = table.loc[valid].reset_index(drop=True)
    tiepoint_tables[core] = table

for core in ['NG1', 'NG2', 'GRIP', 'WDC']:
    table = tiepoint_tables[core].copy()
    age_col = f'{core}_age'
    gaps = np.round(table[age_col].diff(), 3)
    table['Years Since Last Volcanic Event'] = gaps.astype(object)
    table.loc[gaps < 1, 'Years Since Last Volcanic Event'] = 'N/A'
    if not table.empty:
        table.loc[0, 'Years Since Last Volcanic Event'] = 'N/A'
    tiepoint_tables[core] = table

tiepoint_tables['NG1']

,NG1,paired_core_depths,paired_core_name,reference,source_file,pair,NG1_age,Years Since Last Volcanic Event
0,1352.732,1523.784,GISP2,Seierstad2014_Rasmussen2014,GISP2-NG1/Seierstad2014_Rasmussen2014_GISP2-NG...,GISP2-NG1,9600.775,N/A
1,1352.732,1352.313,NG2,Seierstad2014_Rasmussen2014,NG1-NG2/Seierstad2014_Rasmussen2014_NG1-NG2.txt,NG1-NG2,9600.775,N/A
2,1352.732,1465.380,GRIP,Seierstad2014_Rasmussen2014,GRIP-NG1/Seierstad2014_Rasmussen2014_GRIP-NG1.txt,GRIP-NG1,9600.775,N/A
3,1352.740,1323.725,NEEM,NEEM-GICC05,NEEM-NG1/NEEM-GICC05_NEEM-NG1.txt,NEEM-NG1,9600.875,N/A
4,1353.592,1324.304,NEEM,NEEM-GICC05,NEEM-NG1/NEEM-GICC05_NEEM-NG1.txt,NEEM-NG1,9610.886,10.011
5,1354.514,1354.108,NG2,Seierstad2014_Rasmussen2014,NG1-NG2/Seierstad2014_Rasmussen2014_NG1-NG2.txt,NG1-NG2,9620.262,9.376
6,1354.514,1467.339,GRIP,Seierstad2014_Rasmussen2014,GRIP-NG1/Seierstad2014_Rasmussen2014_GRIP-NG1.txt,GRIP-NG1,9620.262,N/A
7,1354.514,1525.729,GISP2,Seierstad2014_Rasmussen2014,GISP2-NG1/Seierstad2014_Rasmussen2014_GISP2-NG...,GISP2-NG1,9620.262,N/A
8,1354.525,1324.965,NEEM,NEEM-GICC05,NEEM-NG1/NEEM-GICC05_NEEM-NG1.txt,NEEM-NG1,9620.346,N/A
9,1354.832,1354.471,NG2,Seierstad2014_Rasmussen2014,NG1-NG2/Seierstad2014_Rasmussen2014_NG1-NG2.txt,NG1-NG2,9624.473,4.127


In [146]:
export_cores = ['NG1', 'NG2', 'GRIP', 'WDC']
output_excel = Path('/Users/quinnmackay/Desktop/tiepoint_tables_by_core.xlsx')

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    for core in export_cores:
        export_df = tiepoint_tables[core].drop(columns=['source_file', 'pair'], errors='ignore')
        export_df.to_excel(writer, sheet_name=core, index=False)

print(f'Exported {len(export_cores)} sheets to: {output_excel}')

Exported 4 sheets to: /Users/quinnmackay/Desktop/tiepoint_tables_by_core.xlsx
